# Agente de Verificación de Listas Escolares - Librería Antártica

#Este Jupyter Notebook integra el flujo multi-paso del agente:
1. Carga de datos de catálogo, stock y lista escolar.
2. Generación de embeddings con Gemini (`models/gemini-embedding-01`) y recuperación semántica (RAG).
3. Consulta de stock exacto por ISBN y sucursal.
4. Orquestación y consolidación de resultados.

In [3]:
# Instalación de las librerías necesarias segun requirements.txt
!pip install numpy google-generativeai python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
from dotenv import load_dotenv

# Cargar variables desde archivo .env si existe
load_dotenv()

# Configurar API Keys
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "TU_GOOGLE_API_KEY_AQUI")
os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "TU_LLM_API_KEY_AQUI")

if os.environ["GOOGLE_API_KEY"] == "TU_GOOGLE_API_KEY_AQUI":
    print("⚠️ Recuerda reemplazar 'TU_GOOGLE_API_KEY_AQUI' con tu API Key válida de Google AI Studio.")

⚠️ Recuerda reemplazar 'TU_GOOGLE_API_KEY_AQUI' con tu API Key válida de Google AI Studio.


In [ ]:
import os
import csv

# Crear carpeta 'data' si no existe
os.makedirs("data", exist_ok=True)

# 1. Muestra de Catálogo
catalogo_data = [
    {"isbn": "9789561234567", "titulo": "Papelucho", "autor": "Marcela Paz", "categoria": "Infantil", "edad_curso": "8 años / 3° Básico"},
    {"isbn": "9789561234568", "titulo": "El Principito", "autor": "Antoine de Saint-Exupéry", "categoria": "Literatura", "edad_curso": "10 años / 5° Básico"},
    {"isbn": "9789561234569", "titulo": "Sub Terra", "autor": "Baldomero Lillo", "categoria": "Narrativa Chilena", "edad_curso": "14 años / 1° Medio"}
]
with open("data/catalogo_muestra.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["isbn", "titulo", "autor", "categoria", "edad_curso"])
    writer.writeheader()
    writer.writerows(catalogo_data)

# 2. Muestra de Stock por Sucursal
stock_data = [
    {"isbn": "9789561234567", "sucursal": "Mall Costanera Center", "cantidad": "5"},
    {"isbn": "9789561234567", "sucursal": "Plaza Vespucio", "cantidad": "0"},
    {"isbn": "9789561234568", "sucursal": "Mall Costanera Center", "cantidad": "2"},
    {"isbn": "9789561234569", "sucursal": "Plaza Vespucio", "cantidad": "0"}
]
with open("data/stock_sucursales.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["isbn", "sucursal", "cantidad"])
    writer.writeheader()
    writer.writerows(stock_data)

# 3. Lista Escolar de Ejemplo
lista_data = [
    {"titulo": "Papelucho", "curso": "3° Básico"},
    {"titulo": "El Principito", "curso": "5° Básico"},
    {"titulo": "Sub Terra", "curso": "1° Medio"}
]
with open("data/lista_escolar_ejemplo.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["titulo", "curso"])
    writer.writeheader()
    writer.writerows(lista_data)

print("✅ Archivos CSV de prueba creados en el directorio 'data/'")

In [6]:
# Plantillas de prompt del agente de atención de Librería Antártica

SYSTEM_PROMPT = """Eres el asistente de atención al cliente de Librería Antártica.
Respondes solo en base al CONTEXTO entregado (catálogo, stock y listas escolares).
Si el dato no está en el contexto, indica que no tienes esa información y ofrece
derivar a un ejecutivo; nunca inventes disponibilidad, precio o título.
Cita siempre la sucursal y la fuente del dato. Responde en español, de forma
breve y cordial."""

PROMPT_DISPONIBILIDAD = """Consulta del cliente: {consulta_usuario}
CONTEXTO recuperado (top-k del catálogo y stock):
{contexto_rag}

Instrucciones: identifica el/los título(s) mencionados, confirma disponibilidad
por sucursal usando solo el CONTEXTO, y si no hay stock sugiere hasta 2
alternativas del mismo género/nivel presentes en el CONTEXTO."""

PROMPT_VERIFICACION_TITULO = """Título requerido: {titulo_lista}, curso: {curso}
CONTEXTO: {resultado_busqueda_catalogo_y_stock}

Devuelve solo un JSON con este formato exacto:
{{"titulo": "...", "disponible": true/false, "sucursal": "...", "cantidad": 0,
"alternativa_si_no_disponible": "..."}}"""

PROMPT_CONSOLIDACION = """A partir de esta lista de resultados por título:
{lista_resultados_json}

Redacta un resumen para el apoderado: cuántos títulos están disponibles, en qué
sucursal retirarlos, y cuáles requieren reemplazo o encargo, en un máximo de
8 líneas."""

In [7]:
from collections import defaultdict

def cargar_stock(path="data/stock_sucursales.csv"):
    stock = defaultdict(list)
    with open(path, newline="", encoding="utf-8") as f:
        for fila in csv.DictReader(f):
            stock[fila["isbn"]].append(
                {"sucursal": fila["sucursal"], "cantidad": int(fila["cantidad"])}
            )
    return stock

def disponibilidad_por_isbn(stock, isbn):
    """Devuelve las sucursales con stock > 0 para un ISBN dado."""
    registros = stock.get(isbn, [])
    return [r for r in registros if r["cantidad"] > 0]

In [8]:
import numpy as np

try:
    import google.generativeai as genai
except ImportError:
    genai = None

EMBED_MODEL = "models/gemini-embedding-001"

def cargar_catalogo(path="data/catalogo_muestra.csv"):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def construir_chunk(fila):
    """Une los campos relevantes de un producto en un solo texto (chunk)."""
    return (
        f"Título: {fila['titulo']}. Autor: {fila['autor']}. "
        f"Categoría: {fila['categoria']}. Edad/curso sugerido: {fila['edad_curso']}. "
        f"ISBN: {fila['isbn']}."
    )

def embed_textos(textos):
    if genai is None:
        raise RuntimeError("Instala google-generativeai (ver requirements.txt)")
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    vectores = []
    for t in textos:
        resp = genai.embed_content(model=EMBED_MODEL, content=t)
        vectores.append(resp["embedding"])
    return np.array(vectores)

def construir_indice(catalogo):
    chunks = [construir_chunk(f) for f in catalogo]
    vectores = embed_textos(chunks)
    return {"catalogo": catalogo, "chunks": chunks, "vectores": vectores}

def buscar(indice, consulta, top_k=3):
    """Recupera los top_k productos más similares a la consulta."""
    vec_consulta = embed_textos([consulta])[0]
    sims = indice["vectores"] @ vec_consulta / (
        np.linalg.norm(indice["vectores"], axis=1) * np.linalg.norm(vec_consulta) + 1e-8
    )
    top_idx = np.argsort(-sims)[:top_k]
    return [indice["catalogo"][i] for i in top_idx]

c:\Users\tatim\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\tatim\AppData\Local\Temp\ipykernel_11440\3533951928.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [9]:
import json

def cargar_lista_escolar(path="data/lista_escolar_ejemplo.csv"):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def verificar_lista(path_lista="data/lista_escolar_ejemplo.csv"):
    catalogo = cargar_catalogo()
    indice = construir_indice(catalogo)
    stock = cargar_stock()
    lista = cargar_lista_escolar(path_lista)

    resultados = []
    for item in lista:
        candidatos = buscar(indice, item["titulo"], top_k=1)
        if not candidatos:
            resultados.append({"titulo": item["titulo"], "disponible": False,
                                "sucursal": None, "cantidad": 0,
                                "alternativa_si_no_disponible": None})
            continue
        producto = candidatos[0]
        disponibles = disponibilidad_por_isbn(stock, producto["isbn"])
        if disponibles:
            mejor = max(disponibles, key=lambda r: r["cantidad"])
            resultados.append({"titulo": producto["titulo"], "disponible": True,
                                "sucursal": mejor["sucursal"], "cantidad": mejor["cantidad"],
                                "alternativa_si_no_disponible": None})
        else:
            resultados.append({"titulo": producto["titulo"], "disponible": False,
                                "sucursal": None, "cantidad": 0,
                                "alternativa_si_no_disponible": "Consultar encargo"})

    return resultados

# Ejecución del agente
resultados = verificar_lista()
print("Resultados consolidados de la verificación:")
print(json.dumps(resultados, indent=2, ensure_ascii=False))

FileNotFoundError: [Errno 2] No such file or directory: 'data/catalogo_muestra.csv'

In [10]:
# Ejemplo de invocación del LLM para consolidar el resumen final usando Gemini
def consolidar_resumen_llm(resultados):
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    model = genai.GenerativeModel("gemini-1.5-flash", system_instruction=SYSTEM_PROMPT)
    
    prompt = PROMPT_CONSOLIDACION.format(
        lista_resultados_json=json.dumps(resultados, ensure_ascii=False)
    )
    
    response = model.generate_content(prompt)
    return response.text

# Generar resumen consolidado
try:
    resumen = consolidar_resumen_llm(resultados)
    print("--- Resumen para el Apoderado ---")
    print(resumen)
except Exception as e:
    print(f"Error al generar el resumen con el LLM: {e}")

Error al generar el resumen con el LLM: name 'resultados' is not defined
